# Apply Burn-In Perturbations Using NRTK

This notebook demonstrates how to use NRTK to add a fake on-screen display to video data.

```{warning}
`MISBST1909BurnInPerturber` is an experimental interface. Its API and behavior may change in future releases. Perturbers must be enabled using `import nrtk.experimental`.
```

To run this notebook in Colab, use the link below:

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Kitware/nrtk/blob/main/docs/examples/misb_st1909_burn_in_perturber.ipynb)

## Set Up the Environment
 
```{note}
We are suppressing warnings within this notebook to reduce visual clutter for demonstration purposes. If any issues arise while executing this notebook, we recommend that this cell is **not** executed so that any related warnings are shown.
```

In [1]:
import warnings

warnings.filterwarnings("ignore")

```{important}
This notebook requires NRTK with the following extra(s):

- `pillow`: The Pillow image processing library for burn-in perturbations
- `pyav`: Python bindings for libav for video loading

The next cell will install these extra(s). See [Installation](<project:/getting_started/installation.rst>) for alternatives.
```

In [2]:
%pip install -qU pip
print("Installing nrtk with required extras...")
%pip install -q "nrtk[pillow,pyav]"
print("Installing notebook-specific packages...")
%pip install -q matplotlib
print("Done!")

from nrtk.utils._extras import print_extras_status  # noqa: E402 - intentionally after %pip install

print_extras_status()

Note: you may need to restart the kernel to use updated packages.
Installing nrtk with required extras...


Note: you may need to restart the kernel to use updated packages.
Installing notebook-specific packages...


Note: you may need to restart the kernel to use updated packages.
Done!
Detected status of NRTK extras and their dependencies:



[albumentations]
  - nrtk-albumentations       ✗ missing

[diffusion]
  - torch                     ✗ missing
  - diffusers                 ✗ missing
  - accelerate                ✗ missing
  - Pillow                    ✓ 12.3.0
  - transformers              ✗ missing
  - protobuf                  ✗ missing

[graphics]
  - opencv-python             ✗ missing

[hcipy]
  - hcipy                     ✗ missing
  - scipy                     ✗ missing
  - xxhash                    ✗ missing

[headless]
  - opencv-python-headless    ✗ missing

[maite]
  - maite                     ✗ missing

[pillow]
  - Pillow                    ✓ 12.3.0

[pyav]
  - av                        ✓ 17.1.0

[pybsm]
  - pybsm                     ✗ missing

[skimage]
  - scikit-image              ✗ missing

[tools]
  - kwcoco                    ✗ missing
  - Pillow                    ✓ 12.3.0
  - click                     ✓ 8.5.0
  - fastapi                   ✗ missing
  - uvicorn                   ✗ missing
  - pyd

    https://nrtk.readthedocs.io/en/stable/



```{note}
**Colab users**: After setting up the environment, you may need to "Restart Runtime" in order to resolve package version conflicts (see the [README](https://github.com/Kitware/nrtk/blob/main/docs/examples/README.md#run-the-notebooks-from-colab) for more info).
```

In [3]:
import os
import sys
from pathlib import Path

# notebook_utils lives in docs/examples (this notebook's parent directory).
for _anc in [Path.cwd(), *Path.cwd().parents]:
    if (_anc / "docs" / "examples" / "notebook_utils").is_dir():
        sys.path.insert(0, str(_anc / "docs" / "examples"))
        break
else:
    raise ImportError("Cannot locate notebook_utils")

from notebook_utils.video import (  # pyright: ignore[reportMissingImports] # noqa: E402 - intentionally after %pip install
    read_video,
    video_comparison,
)

import nrtk.experimental  # noqa: F401, E402

## Setup: Source Video

In the next cell, we'll load and display a source video from the HMIE dataset. The dataset will be cached in a local `data` subdirectory.

In [4]:
import glob
import urllib.request
import zipfile

NUM_FRAMES = 120

# The HMIE dataset (video + tracks) lives in this notebook's data subdirectory.
data_dir = "./data"
os.makedirs(data_dir, exist_ok=True)
HMIE_PATH = os.path.join(data_dir, "highway_hmie")
if not os.path.isdir(HMIE_PATH):
    # The Girder folder download streams a zip whose root folder is
    # highway_hmie/, so extracting into data_dir creates HMIE_PATH.
    url = "https://data.kitware.com/api/v1/folder/6a67b8461866d4fa1981b9be/download"
    zip_path = os.path.join(data_dir, "highway_hmie.zip")
    _ = urllib.request.urlretrieve(url, zip_path)
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(data_dir)
    os.remove(zip_path)


# Load the source video straight from the HMIE dataset, so the notebook is
# self-contained and never has to regenerate it.
video_path = glob.glob(os.path.join(HMIE_PATH, "**", "seq_*", "*.mp4"), recursive=True)[0]
print(f"Using video from HMIE dataset: {video_path}")

video_frames = read_video(video_path, max_frames=NUM_FRAMES)

print(f"Loaded {len(video_frames)} frames, shape: {video_frames[0].image.shape}, dtype: {video_frames[0].image.dtype}")
dt_ms = int(1000 * (video_frames[1].timestamp - video_frames[0].timestamp)) if len(video_frames) > 1 else 42

video_comparison(
    frame_sequences=[
        [f.image for f in video_frames],
    ],
    titles=["Original Video"],
    interval=dt_ms,
)

Using video from HMIE dataset: ./data/highway_hmie/out_000000_000000/out_000000_000001/seq_mp4/out_000000_000001.mp4


Loaded 120 frames, shape: (720, 1280, 3), dtype: uint8


## Perturbation Gallery

The [MISB ST1909 Burn-In perturber](<project:/development/experimental.rst#nrtk.impls.perturb_video.burn_in.MISBST1909BurnInPerturber>) adds text and other decals to a video in line with the Motion Imagery Standards Board's [ST1909](https://nsgreg.nga.mil/doc/view?i=5096) document. This is done to mimic video data sources which come with metadata "burned in" to the pixels of each video frame in a similar manner.

In [5]:
from nrtk.impls.perturb_video.burn_in import MISBST1909BurnInPerturber

configurations = [
    {
        "regenerate_metadata": False,  # Do not refresh random metadata values each frame
    },
    {
        "color": (0, 0, 0),  # Set text and decal color to black
        "outline_color": None,  # Omit outline around text and decals
        "text_size": 1 / 24,  # Set text size to 1/24th the height of the frame
        "line_thickness": 1,  # Set thickness of decal lines to 1 pixel
    },
]
frame_sequences = []
for configuration in configurations:
    perturber = MISBST1909BurnInPerturber(**configuration)
    frame_sequences.append([frame.image for frame in perturber(frames=iter(video_frames))])

video_comparison(
    frame_sequences=frame_sequences,
    titles=["Random Static Burn-In", "Random Dynamic Burn-In"],
    interval=dt_ms,
)